In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
MOVES_CSV_PATH = "moves.csv"
META_CSV_PATH = "gameinfo.csv"

OUTPUT_DIR = Path("sampled_xiangqi_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

LOW_ELO_MAX = 1100
MID_ELO_MIN = 1100
MID_ELO_MAX = 1400

N_LOW = 1000
N_MID = 1000

RANDOM_SEED = 42

ALL_GAMES_OUTPUT_TXT = OUTPUT_DIR / "human_games_all_uci.txt"
ALL_GAMES_OUTPUT_JSONL = OUTPUT_DIR / "human_games_all_uci.jsonl"
ALL_GAMES_OUTPUT_CSV = OUTPUT_DIR / "human_games_all_metadata.csv"
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
moves_df = pd.read_csv(MOVES_CSV_PATH)
meta_df = pd.read_csv(META_CSV_PATH)

print("moves_df shape:", moves_df.shape)
print("meta_df shape :", meta_df.shape)

display(moves_df.head())
display(meta_df.head())

In [ ]:
moves_df = moves_df.copy()
meta_df = meta_df.copy()

moves_df["gameID"] = moves_df["gameID"].astype(str).str.strip()
moves_df["turn"] = pd.to_numeric(moves_df["turn"], errors="coerce")
moves_df["side"] = moves_df["side"].astype(str).str.strip().str.lower()
moves_df["move"] = moves_df["move"].astype(str).str.strip()

meta_df["gameID"] = meta_df["gameID"].astype(str).str.strip()
meta_df["blackELO"] = pd.to_numeric(meta_df["blackELO"], errors="coerce")
meta_df["redELO"] = pd.to_numeric(meta_df["redELO"], errors="coerce")
meta_df["winner"] = meta_df["winner"].astype(str).str.strip().str.lower()

moves_df = moves_df.dropna(subset=["gameID", "turn", "side", "move"]).copy()
moves_df["turn"] = moves_df["turn"].astype(int)

meta_df = meta_df.dropna(subset=["gameID", "blackELO", "redELO"]).copy()
meta_df["avg_elo"] = (meta_df["blackELO"] + meta_df["redELO"]) / 2.0

print(meta_df["avg_elo"].describe())

In [ ]:
def interleave_game_rows(game_df: pd.DataFrame):
    red = game_df[game_df["side"] == "red"].sort_values("turn")
    black = game_df[game_df["side"] == "black"].sort_values("turn")

    red = red.drop_duplicates(subset=["turn"], keep="first")
    black = black.drop_duplicates(subset=["turn"], keep="first")

    red_map = {int(row["turn"]): row for _, row in red.iterrows()}
    black_map = {int(row["turn"]): row for _, row in black.iterrows()}

    max_turn = max(
        max(red_map.keys()) if len(red_map) else 0,
        max(black_map.keys()) if len(black_map) else 0
    )

    seq = []
    for t in range(1, max_turn + 1):
        if t in red_map:
            seq.append({"turn": t, "side": "red", "move": str(red_map[t]["move"])})
        if t in black_map:
            seq.append({"turn": t, "side": "black", "move": str(black_map[t]["move"])})
    return seq

In [ ]:
FILES = "abcdefghi"

def side_of_piece(p):
    if p == ".":
        return None
    return "red" if p.isupper() else "black"

def piece_type(p):
    return p.upper()

def initial_board():
    board = [["." for _ in range(9)] for _ in range(10)]

    # Red pieces (bottom)
    board[0] = list("RHEAKAEHR")
    board[2][1] = "C"
    board[2][7] = "C"
    for x in [0, 2, 4, 6, 8]:
        board[3][x] = "P"

    # Black pieces (top)
    board[9] = list("rheakaehr")
    board[7][1] = "c"
    board[7][7] = "c"
    for x in [0, 2, 4, 6, 8]:
        board[6][x] = "p"

    return board

def clone_board(board):
    return [row[:] for row in board]

def in_board(x, y):
    return 0 <= x < 9 and 0 <= y < 10

def palace(side, x, y):
    if side == "red":
        return 3 <= x <= 5 and 0 <= y <= 2
    return 3 <= x <= 5 and 7 <= y <= 9

def crosses_river(side, y):
    return y >= 5 if side == "red" else y <= 4

In [ ]:
HORSE_STEPS = [
    ((1, 2), (0, 1)),
    ((-1, 2), (0, 1)),
    ((2, 1), (1, 0)),
    ((2, -1), (1, 0)),
    ((1, -2), (0, -1)),
    ((-1, -2), (0, -1)),
    ((-2, 1), (-1, 0)),
    ((-2, -1), (-1, 0)),
]

DIRS4 = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def generate_pseudo_moves(board, side):
    moves = []

    def add_move(x, y, nx, ny):
        if not in_board(nx, ny):
            return
        target = board[ny][nx]
        if target != "." and side_of_piece(target) == side:
            return
        moves.append((x, y, nx, ny))

    for y in range(10):
        for x in range(9):
            p = board[y][x]
            if p == "." or side_of_piece(p) != side:
                continue

            pt = piece_type(p)

            if pt == "P":
                dirs = [(0, 1)] if side == "red" else [(0, -1)]
                if crosses_river(side, y):
                    dirs += [(-1, 0), (1, 0)]
                for dx, dy in dirs:
                    add_move(x, y, x + dx, y + dy)

            elif pt == "K":
                for dx, dy in DIRS4:
                    nx, ny = x + dx, y + dy
                    if palace(side, nx, ny):
                        add_move(x, y, nx, ny)

            elif pt == "A":
                for dx, dy in [(1, 1), (-1, 1), (1, -1), (-1, -1)]:
                    nx, ny = x + dx, y + dy
                    if palace(side, nx, ny):
                        add_move(x, y, nx, ny)

            elif pt == "E":
                for dx, dy in [(2, 2), (-2, 2), (2, -2), (-2, -2)]:
                    nx, ny = x + dx, y + dy
                    ex, ey = x + dx // 2, y + dy // 2
                    if not in_board(nx, ny):
                        continue
                    if board[ey][ex] != ".":
                        continue
                    if side == "red" and ny > 4:
                        continue
                    if side == "black" and ny < 5:
                        continue
                    add_move(x, y, nx, ny)

            elif pt == "H":
                for (dx, dy), (lx, ly) in HORSE_STEPS:
                    legx, legy = x + lx, y + ly
                    nx, ny = x + dx, y + dy
                    if in_board(nx, ny) and in_board(legx, legy) and board[legy][legx] == ".":
                        add_move(x, y, nx, ny)

            elif pt in ("R", "C"):
                for dx, dy in DIRS4:
                    nx, ny = x + dx, y + dy
                    jumped = False
                    while in_board(nx, ny):
                        target = board[ny][nx]

                        if pt == "R":
                            if target == ".":
                                moves.append((x, y, nx, ny))
                            else:
                                if side_of_piece(target) != side:
                                    moves.append((x, y, nx, ny))
                                break

                        else:  # Cannon
                            if not jumped:
                                if target == ".":
                                    moves.append((x, y, nx, ny))
                                else:
                                    jumped = True
                            else:
                                if target != ".":
                                    if side_of_piece(target) != side:
                                        moves.append((x, y, nx, ny))
                                    break

                        nx += dx
                        ny += dy

    return moves

In [ ]:
def attacks_square(board, side, tx, ty):
    for y in range(10):
        for x in range(9):
            p = board[y][x]
            if p == "." or side_of_piece(p) != side:
                continue

            pt = piece_type(p)

            if pt == "P":
                dirs = [(0, 1)] if side == "red" else [(0, -1)]
                if crosses_river(side, y):
                    dirs += [(-1, 0), (1, 0)]
                for dx, dy in dirs:
                    if (x + dx, y + dy) == (tx, ty):
                        return True

            elif pt == "H":
                for (dx, dy), (lx, ly) in HORSE_STEPS:
                    legx, legy = x + lx, y + ly
                    nx, ny = x + dx, y + dy
                    if in_board(nx, ny) and board[legy][legx] == "." and (nx, ny) == (tx, ty):
                        return True

            elif pt == "R":
                for dx, dy in DIRS4:
                    nx, ny = x + dx, y + dy
                    while in_board(nx, ny):
                        target = board[ny][nx]
                        if (nx, ny) == (tx, ty):
                            return True
                        if target != ".":
                            break
                        nx += dx
                        ny += dy

            elif pt == "C":
                for dx, dy in DIRS4:
                    nx, ny = x + dx, y + dy
                    jumped = False
                    while in_board(nx, ny):
                        target = board[ny][nx]
                        if not jumped:
                            if target != ".":
                                jumped = True
                        else:
                            if target != ".":
                                if (nx, ny) == (tx, ty) and side_of_piece(target) != side:
                                    return True
                                break
                        nx += dx
                        ny += dy

            elif pt == "K":
                # flying general attack only
                for dx, dy in DIRS4:
                    nx, ny = x + dx, y + dy
                    while in_board(nx, ny):
                        target = board[ny][nx]
                        if target != ".":
                            if (nx, ny) == (tx, ty) and piece_type(target) == "K" and side_of_piece(target) != side:
                                return True
                            break
                        nx += dx
                        ny += dy

    return False

def is_in_check(board, side):
    king_piece = "K" if side == "red" else "k"
    king_pos = None

    for y in range(10):
        for x in range(9):
            if board[y][x] == king_piece:
                king_pos = (x, y)
                break
        if king_pos is not None:
            break

    if king_pos is None:
        return True

    enemy = "black" if side == "red" else "red"
    return attacks_square(board, enemy, king_pos[0], king_pos[1])

def apply_move_copy(board, move):
    x, y, nx, ny = move
    new_board = clone_board(board)
    new_board[ny][nx] = new_board[y][x]
    new_board[y][x] = "."
    return new_board

def apply_move_inplace(board, move):
    x, y, nx, ny = move
    board[ny][nx] = board[y][x]
    board[y][x] = "."

def generate_legal_moves(board, side):
    legal = []
    for move in generate_pseudo_moves(board, side):
        new_board = apply_move_copy(board, move)
        if not is_in_check(new_board, side):
            legal.append(move)
    return legal

In [ ]:
def file_num(side, x):
    return 9 - x if side == "red" else x + 1

def is_forward(side, dy):
    return dy > 0 if side == "red" else dy < 0

def move_to_axf(board, side, move):
    x, y, nx, ny = move
    p = board[y][x]
    pt = piece_type(p)

    same_file_same_type = []
    for yy in range(10):
        pp = board[yy][x]
        if pp != "." and side_of_piece(pp) == side and piece_type(pp) == pt:
            same_file_same_type.append(yy)

    if len(same_file_same_type) == 1:
        prefix = str(file_num(side, x))
    elif len(same_file_same_type) == 2:
        if side == "red":
            prefix = "+" if y == max(same_file_same_type) else "-"
        else:
            prefix = "+" if y == min(same_file_same_type) else "-"
    else:
        ordered = sorted(same_file_same_type, reverse=(side == "red"))
        prefix = str(ordered.index(y) + 1)

    if y == ny:
        op = "."
        arg = str(file_num(side, nx))
    else:
        op = "+" if is_forward(side, ny - y) else "-"
        if pt in ("R", "C", "K", "P"):
            arg = str(abs(ny - y))
        else:
            arg = str(file_num(side, nx))

    return f"{pt}{prefix}{op}{arg}"

def normalize_axf(move_text):
    return str(move_text).strip().upper().replace("=", ".")

def move_to_uci(move):
    x, y, nx, ny = move
    return f"{FILES[x]}{y}{FILES[nx]}{ny}"

def match_raw_move(board, side, raw_move):
    raw_move = normalize_axf(raw_move)
    legal_moves = generate_legal_moves(board, side)

    matches = []
    for move in legal_moves:
        if move_to_axf(board, side, move) == raw_move:
            matches.append(move)

    return matches

In [ ]:
def build_all_game_metadata(meta_df: pd.DataFrame) -> pd.DataFrame:
    meta_small = meta_df[[
        "gameID", "game_datetime", "blackID", "blackELO", "redID", "redELO", "winner", "avg_elo"
    ]].drop_duplicates(subset=["gameID"]).copy()

    def elo_band_from_avg(avg_elo: float) -> str:
        if pd.isna(avg_elo):
            return "unknown"
        if avg_elo < LOW_ELO_MAX:
            return "low"
        if MID_ELO_MIN <= avg_elo <= MID_ELO_MAX:
            return "mid"
        return "high"

    meta_small["elo_band"] = meta_small["avg_elo"].apply(elo_band_from_avg)
    return meta_small

In [ ]:
def convert_all_games_to_uci(moves_df: pd.DataFrame, meta_df: pd.DataFrame):
    all_meta = build_all_game_metadata(meta_df)
    all_game_ids = set(all_meta["gameID"].astype(str))

    all_moves = moves_df[moves_df["gameID"].isin(all_game_ids)].copy()

    converted_games = []
    failed_games = []

    for game_id, game_rows in all_moves.groupby("gameID", sort=False):
        meta_match = all_meta[all_meta["gameID"] == game_id]
        if len(meta_match) == 0:
            failed_games.append({
                "game_id": game_id,
                "reason": {"type": "missing_metadata"}
            })
            continue

        meta_row = meta_match.iloc[0]
        seq = interleave_game_rows(game_rows)

        board = initial_board()
        uci_moves = []
        failed = False
        fail_reason = None

        for ply_idx, item in enumerate(seq, start=1):
            side = item["side"]
            raw_move = item["move"]

            matches = match_raw_move(board, side, raw_move)

            if len(matches) != 1:
                failed = True
                fail_reason = {
                    "ply": ply_idx,
                    "side": side,
                    "raw_move": raw_move,
                    "num_matches": len(matches)
                }
                break

            move = matches[0]
            uci_moves.append(move_to_uci(move))
            apply_move_inplace(board, move)

        if failed:
            failed_games.append({
                "game_id": game_id,
                "reason": fail_reason
            })
            continue

        converted_games.append({
            "game_id": str(game_id),
            "elo_band": str(meta_row["elo_band"]),
            "avg_elo": (
                int(round(float(meta_row["avg_elo"])))
                if not pd.isna(meta_row["avg_elo"]) else -1
            ),
            "red": str(meta_row["redID"]),
            "black": str(meta_row["blackID"]),
            "winner": str(meta_row["winner"]),
            "game_datetime": str(meta_row["game_datetime"]),
            "uci_moves": uci_moves,
            "num_plies": len(uci_moves),
        })

    return converted_games, failed_games

def moves_to_pair_lines(uci_moves):
    chunks = []
    move_no = 1
    i = 0
    while i < len(uci_moves):
        if i + 1 < len(uci_moves):
            chunks.append(f"{move_no}. {uci_moves[i]} {uci_moves[i+1]}")
            i += 2
        else:
            chunks.append(f"{move_no}. {uci_moves[i]}")
            i += 1
        move_no += 1
    return " ".join(chunks)

def save_all_games_outputs(converted_games, failed_games):
    # TXT export (same style as your old output)
    with open(ALL_GAMES_OUTPUT_TXT, "w", encoding="utf-8") as f:
        for game in converted_games:
            f.write(f'[Human Game EloBand "{game["elo_band"]}" AvgElo "{game["avg_elo"]}"]\n')
            #f.write(f'[GameID "{game["game_id"]}"]\n')
            f.write(f'[Red "{game["red"]}"]\n')
            f.write(f'[Black "{game["black"]}"]\n')
            #f.write(f'[Winner "{game["winner"]}"]\n')
            #f.write(f'[DateTime "{game["game_datetime"]}"]\n')
            f.write("\n")
            f.write(moves_to_pair_lines(game["uci_moves"]))
            f.write("\n\n")
    
    with open(ALL_GAMES_OUTPUT_JSONL, "w", encoding="utf-8") as f:
        for game in converted_games:
            f.write(json.dumps(game, ensure_ascii=False) + "\n")
    
    pd.DataFrame([
        {
            "game_id": g["game_id"],
            "elo_band": g["elo_band"],
            "avg_elo": g["avg_elo"],
            "red": g["red"],
            "black": g["black"],
            "winner": g["winner"],
            "game_datetime": g["game_datetime"],
            "num_plies": g["num_plies"],
            "uci_moves": " ".join(g["uci_moves"]),
        }
        for g in converted_games
    ]).to_csv(ALL_GAMES_OUTPUT_CSV, index=False)

    print("saved txt   :", ALL_GAMES_OUTPUT_TXT)
    print("saved jsonl :", ALL_GAMES_OUTPUT_JSONL)
    print("saved csv   :", ALL_GAMES_OUTPUT_CSV)

    if len(failed_games) > 0:
        failed_path = OUTPUT_DIR / "human_games_all_failed.csv"
        pd.DataFrame(failed_games).to_csv(failed_path, index=False)
        print("saved failed:", failed_path)


In [ ]:
all_meta = build_all_game_metadata(meta_df)
print("total games in metadata:", len(all_meta))

converted_games, failed_games = convert_all_games_to_uci(moves_df, meta_df)

print("converted games:", len(converted_games))
print("failed games   :", len(failed_games))

if len(failed_games) > 0:
    display(pd.DataFrame(failed_games).head())

save_all_games_outputs(converted_games, failed_games)